# GodelEnv Training Notebook

This notebook trains a model on the GodelEnv recursive self-improvement environment.

**Pipeline:**
1. Install dependencies and clone repo
2. Configure API keys (optional, for LLM-backed grading)
3. Collect prompts from the environment
4. SFT warm-start on demonstration data
5. GRPO refinement with reward functions
6. Evaluate and export metrics

**Evidence Quality:**
- `deterministic` mode: Fast, reproducible, uses heuristic graders
- `auto` mode: Uses LLM for grading (requires API keys), stronger evidence

In [ ]:
# Cell 1: Setup - Clone repo and install dependencies
import os
import subprocess
import sys
from pathlib import Path

# Check if running in Colab
IN_COLAB = "COLAB_RELEASE_TAG" in os.environ

if IN_COLAB:
    repo_dir = Path("/content/GodelEnv")
    if not repo_dir.exists():
        !git clone https://github.com/dwan-ith/GodelEnv.git /content/GodelEnv
    os.chdir(repo_dir)
    !pip install -q -e ".[train]"
    
print(f"Working directory: {os.getcwd()}")

In [ ]:
# Cell 2: API Keys Configuration (OPTIONAL - run BEFORE other cells)
# 
# To use LLM-backed grading (stronger evidence), set your API key here.
# IMPORTANT: The variable name must be EXACTLY 'OPENAI_API_KEY' or 'HF_TOKEN'
#
# Option 1: OpenAI API
# os.environ["OPENAI_API_KEY"] = "sk-..."  # Your OpenAI API key
#
# Option 2: Hugging Face (using your $30 credits)
# os.environ["HF_TOKEN"] = "hf_..."  # Your HF token from huggingface.co/settings/tokens
#
# In Colab, you can also use Secrets:
# from google.colab import userdata
# os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
# os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')

import os

# Uncomment ONE of these to enable LLM grading:
# os.environ["OPENAI_API_KEY"] = "your-key-here"
# os.environ["HF_TOKEN"] = "your-hf-token-here"

# Check what's configured
has_openai = bool(os.environ.get("OPENAI_API_KEY"))
has_hf = bool(os.environ.get("HF_TOKEN"))
print(f"OpenAI API configured: {has_openai}")
print(f"HuggingFace token configured: {has_hf}")
if not has_openai and not has_hf:
    print("\n⚠️  No API keys set - will use deterministic mode (faster but weaker evidence)")
else:
    print("\n✓ API key detected - can use 'auto' mode for LLM-backed grading")

In [ ]:
# Cell 3: Configuration
import os
import json
import random
import warnings
from pathlib import Path

import numpy as np
import torch
from transformers import set_seed

# Silence noisy warnings
os.environ["TRL_EXPERIMENTAL_SILENCE"] = "1"
warnings.filterwarnings("ignore", category=UserWarning, module="trl")

# Training configuration
CONFIG = {
    "tasks": ["factual_qa", "alignment_qa", "reasoning", "strategy_optimization"],
    "num_prompts": 16,
    "sft_steps": 20,
    "grpo_steps": 10,
    "max_input_length": 512,
    "max_new_tokens": 256,
    "batch_size": 2,
    "seed": 42,
    # Change to 'auto' if you have API keys configured above
    "grading_mode": "deterministic",
    "strategy_eval_mode": "deterministic",
}

OUTPUT_DIR = Path("artifacts/training_run")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Set environment for grading mode
os.environ["GODEL_GRADING_MODE"] = CONFIG["grading_mode"]
os.environ["GODEL_STRATEGY_EVAL_MODE"] = CONFIG["strategy_eval_mode"]

# Set seeds for reproducibility
random.seed(CONFIG["seed"])
np.random.seed(CONFIG["seed"])
torch.manual_seed(CONFIG["seed"])
set_seed(CONFIG["seed"])

# Detect GPU
USE_GPU = torch.cuda.is_available()
print(f"Using GPU: {USE_GPU}")
if USE_GPU:
    print(f"GPU: {torch.cuda.get_device_name(0)}")

print(f"\nConfiguration:")
for k, v in CONFIG.items():
    print(f"  {k}: {v}")

In [ ]:
# Cell 4: Import GodelEnv modules
from godel_engine.rollout import collect_local_prompt_dataset
from godel_engine.training_support import (
    build_supervised_examples_freeform,
    build_freeform_model,
    evaluate_model_freeform,
    load_tokenizer,
    plot_before_after,
    plot_training_curves,
    run_grpo,
    run_sft,
    warn_evidence_quality,
)

# Check evidence quality
evidence_quality = warn_evidence_quality(
    generation_mode="freeform",
    grading_mode=CONFIG["grading_mode"],
    strategy_eval_mode=CONFIG["strategy_eval_mode"],
)
print(f"\nEvidence quality: {evidence_quality}")

## 2. Collect Training Data

In [ ]:
# Collect prompts from the environment
print("Collecting prompts from GodelEnv...")
prompt_data = collect_local_prompt_dataset(
    num_prompts=CONFIG["num_prompts"],
    tasks=CONFIG["tasks"],
    seed=CONFIG["seed"],
)

print(f"Collected {len(prompt_data)} prompts")
print(f"Task distribution:")
from collections import Counter
task_counts = Counter(p["task_type"] for p in prompt_data)
for task, count in sorted(task_counts.items()):
    print(f"  {task}: {count}")

In [ ]:
# Build supervised examples for SFT warm-start
print("Building supervised examples...")
supervised_examples = build_supervised_examples_freeform(prompt_data)
print(f"Created {len(supervised_examples)} SFT examples")

# Show example prompt/completion
print("\n--- Example SFT pair ---")
print(f"Prompt (first 300 chars):\n{supervised_examples[0]['prompt'][:300]}...")
print(f"\nCompletion (first 200 chars):\n{supervised_examples[0]['completion'][:200]}...")

## 3. Initialize Model

In [ ]:
# Load tokenizer and build model
print("Loading tokenizer...")
tokenizer = load_tokenizer()

print("Building model for JSON generation...")
model = build_freeform_model(
    tokenizer,
    max_length=CONFIG["max_input_length"] + CONFIG["max_new_tokens"] + 256,
)

# Move to GPU if available
if USE_GPU:
    model = model.cuda()

print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Model device: {next(model.parameters()).device}")

## 4. Baseline Evaluation

In [ ]:
# Evaluate baseline (random policy)
print("Evaluating baseline (random generation)...")
baseline_metrics = evaluate_model_freeform(
    model,
    tokenizer,
    prompt_data,
    max_new_tokens=CONFIG["max_new_tokens"],
    max_input_length=CONFIG["max_input_length"],
    policy_mode="random",
    seed=CONFIG["seed"],
)

print(f"\nBaseline metrics:")
print(f"  Mean reward: {baseline_metrics['mean_reward']:.4f}")
print(f"  Mean score: {baseline_metrics['mean_score']:.4f}")
print(f"  JSON action rate: {baseline_metrics.get('json_action_rate', 0):.1%}")
print(f"  Strategy patch rate: {baseline_metrics.get('strategy_patch_rate', 0):.1%}")

## 5. SFT Warm-Start

In [ ]:
# Run SFT warm-start
print(f"Running SFT for {CONFIG['sft_steps']} steps...")
sft_logs = run_sft(
    model,
    tokenizer,
    supervised_examples,
    output_dir=OUTPUT_DIR / "sft",
    max_steps=CONFIG["sft_steps"],
    batch_size=1,
    max_length=CONFIG["max_input_length"] + CONFIG["max_new_tokens"],
    use_cpu=not USE_GPU,
)

# Show loss progression
losses = [log["loss"] for log in sft_logs if "loss" in log]
if losses:
    print(f"\nSFT Loss: {losses[0]:.4f} -> {losses[-1]:.4f}")

## 6. GRPO Refinement

In [ ]:
# Run GRPO refinement
print(f"Running GRPO for {CONFIG['grpo_steps']} steps...")
grpo_logs = run_grpo(
    model,
    tokenizer,
    prompt_data,
    output_dir=OUTPUT_DIR / "grpo",
    max_steps=CONFIG["grpo_steps"],
    batch_size=CONFIG["batch_size"],
    num_generations=2,
    max_completion_length=CONFIG["max_new_tokens"],
    max_new_tokens=CONFIG["max_new_tokens"],
    use_cpu=not USE_GPU,
)

# Show reward progression
rewards = [log["reward"] for log in grpo_logs if "reward" in log]
if rewards:
    print(f"\nGRPO Reward: {rewards[0]:.4f} -> {rewards[-1]:.4f}")

## 7. Final Evaluation

In [ ]:
# Evaluate trained model
print("Evaluating trained model...")
trained_metrics = evaluate_model_freeform(
    model,
    tokenizer,
    prompt_data,
    max_new_tokens=CONFIG["max_new_tokens"],
    max_input_length=CONFIG["max_input_length"],
    policy_mode="model",
    seed=CONFIG["seed"],
)

print(f"\nTrained metrics:")
print(f"  Mean reward: {trained_metrics['mean_reward']:.4f}")
print(f"  Mean score: {trained_metrics['mean_score']:.4f}")
print(f"  JSON action rate: {trained_metrics.get('json_action_rate', 0):.1%}")
print(f"  Strategy patch rate: {trained_metrics.get('strategy_patch_rate', 0):.1%}")

# Compute improvement
reward_delta = trained_metrics["mean_reward"] - baseline_metrics["mean_reward"]
score_delta = trained_metrics["mean_score"] - baseline_metrics["mean_score"]
print(f"\nImprovement:")
print(f"  Reward: {'+' if reward_delta >= 0 else ''}{reward_delta:.4f}")
print(f"  Score: {'+' if score_delta >= 0 else ''}{score_delta:.4f}")

## 8. Generate Plots and Save Results

In [ ]:
# Generate training curves
print("Generating plots...")
plots = plot_training_curves(sft_logs, grpo_logs, OUTPUT_DIR)
before_after_path = plot_before_after(baseline_metrics, trained_metrics, OUTPUT_DIR)

print(f"Saved plots to {OUTPUT_DIR}")

In [ ]:
# Display plots
from IPython.display import Image, display

print("Loss Curve:")
display(Image(filename=str(plots["loss_curve"])))

print("\nReward Curve:")
display(Image(filename=str(plots["reward_curve"])))

print("\nBefore/After Comparison:")
display(Image(filename=str(before_after_path)))

In [ ]:
# Save model and metrics
print("Saving model and metrics...")

# Save model
model_dir = OUTPUT_DIR / "final_model"
model_dir.mkdir(parents=True, exist_ok=True)
model.save_pretrained(model_dir)
tokenizer.save_pretrained(model_dir)

# Save metrics summary
summary = {
    "config": CONFIG,
    "evidence_quality": evidence_quality,
    "baseline": {
        "mean_reward": baseline_metrics["mean_reward"],
        "mean_score": baseline_metrics["mean_score"],
        "json_action_rate": baseline_metrics.get("json_action_rate", 0),
        "strategy_patch_rate": baseline_metrics.get("strategy_patch_rate", 0),
    },
    "trained": {
        "mean_reward": trained_metrics["mean_reward"],
        "mean_score": trained_metrics["mean_score"],
        "json_action_rate": trained_metrics.get("json_action_rate", 0),
        "strategy_patch_rate": trained_metrics.get("strategy_patch_rate", 0),
    },
    "improvement": {
        "reward_delta": reward_delta,
        "score_delta": score_delta,
    },
    "artifacts": {
        "loss_curve": str(plots["loss_curve"]),
        "reward_curve": str(plots["reward_curve"]),
        "before_after": str(before_after_path),
        "model_dir": str(model_dir),
    },
}

summary_path = OUTPUT_DIR / "metrics.json"
summary_path.write_text(json.dumps(summary, indent=2))
print(f"Saved metrics to {summary_path}")

print("\n" + "="*50)
print("TRAINING COMPLETE")
print("="*50)
print(f"Evidence quality: {evidence_quality}")
print(f"Reward improvement: {'+' if reward_delta >= 0 else ''}{reward_delta:.4f}")
print(f"Score improvement: {'+' if score_delta >= 0 else ''}{score_delta:.4f}")

## Summary

This notebook trained a model on GodelEnv, demonstrating:

1. **Data Collection**: Prompts collected from the recursive self-improvement environment
2. **SFT Warm-Start**: Model initialized with demonstration data
3. **GRPO Refinement**: Policy improved via reward-based optimization
4. **Evaluation**: Before/after comparison showing improvement

For stronger evidence, set `grading_mode` and `strategy_eval_mode` to `"auto"` and provide API keys in Cell 2.